In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from thermohaline.detect_staircases import classify_staircase, identify_staircases_from_layers
plt.rcParams.update({'font.size': 14})

# Thermohaline classifier demo

This notebook demonstrates the functionality of the staircase classification algorithm

### Test classifier with synthetic data

First we create synthetic profiles of temperature and salinity data to check that the classifier works on idealised data. This is similar to the [internal software test](https://github.com/callumrollo/thermohaline-steps/blob/main/test_staircase.py) used in development.

Note that the classifier can work on any evenly spaced data. e.g. try changing `pressure_step` to 0.01 and `pressure_range` to [0, 10] to see how it works at small scales

In [ ]:
pressure_step = 1
pressure_range = [0, 1000]
diff_c = -1
diff_s = -0.1
mix = 80
interface = 5
current_c = 20
current_s = 30
buffer_length = 150

p_in = np.arange(pressure_range[0], pressure_range[1] + pressure_step, pressure_step)
ct_in = np.empty(len(p_in))
sa_in = np.empty(len(p_in))


ct_in[:buffer_length] = current_c
sa_in[:buffer_length] = current_s
for start in np.arange(buffer_length, len(p_in) - buffer_length, mix+interface):
    ct_in[start: start + interface] = np.linspace(current_c, current_c + diff_c, interface)
    current_c +=diff_c
    ct_in[start + interface: start + interface + mix] = current_c
    sa_in[start: start + interface] = np.linspace(current_s, current_s + diff_s, interface)
    current_s +=diff_s
    sa_in[start + interface: start + interface + mix] = current_s  

ct_in[len(p_in)-buffer_length:] = current_c
sa_in[len(p_in)-buffer_length:] = current_s

fig, ax = plt.subplots(1,2, figsize=(12,8), sharey="row")
ax = ax.ravel()
ax[0].plot(ct_in, p_in, color="k", alpha=0.5)
ax[0].set(xlabel="Conservative temperature (C)", ylabel="Pressure (dbar)")
ax[1].plot(sa_in, p_in, color="k", alpha=0.5)
ax[1].set(xlabel="Absolute salinity (g/kg)")
ax[0].invert_yaxis()


Next we run the classifier on this synthetic data and plot the results

In [ ]:
df, mixes, grads = classify_staircase(p_in, ct_in, sa_in, show_steps=True)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,8), sharey="row")
ax = ax.ravel()
ax[0].plot(ct_in, p_in, color="k", alpha=0.2)
ax[0].set(xlabel="Conservative temperature (C)", ylabel="Pressure (dbar)")
ax[1].plot(sa_in, p_in, color="k", alpha=0.2)
ax[1].set(xlabel="Absolute salinity (g/kg)")

ax[0].plot(np.ma.array(df.ct, mask=df['mixed_layer_final_mask']), df.p, color='C0', label="Mixed layer")
ax[0].plot(np.ma.array(df.ct, mask=df['gradient_layer_final_mask']), df.p, color='C1', label="Interface")
ax[1].plot(np.ma.array(df.sa, mask=df['mixed_layer_final_mask']), df.p, color='C0')
ax[1].plot(np.ma.array(df.sa, mask=df['gradient_layer_final_mask']), df.p, color='C1')

ax[0].legend()
ax[0].invert_yaxis()

Due to the use of a rolling mean window to calculate background gradients, the upper and lower ends of the profile are not classified

### Experimentation

We can turn the above process into a function and re-run it with different input data

In [ ]:
def classify_test_data(p, ct, sa, temp_flag_only=False):
    df, mixes, grads = classify_staircase(p, ct, sa, temp_flag_only=temp_flag_only)
    fig, ax = plt.subplots(1,2, figsize=(12,8), sharey="row")
    ax = ax.ravel()
    ax[0].plot(ct, p, color="k", alpha=0.2)
    ax[0].set(xlabel="Conservative temperature (C)", ylabel="Pressure (dbar)")
    ax[1].plot(sa, p, color="k", alpha=0.2)
    ax[1].set(xlabel="Absolute salinity (g/kg)")

    ax[0].plot(np.ma.array(df.ct, mask=df['mixed_layer_final_mask']), df.p, color='C0', label="Mixed layer")
    ax[0].plot(np.ma.array(df.ct, mask=df['gradient_layer_final_mask']), df.p, color='C1', label="Interface")
    ax[1].plot(np.ma.array(df.sa, mask=df['mixed_layer_final_mask']), df.p, color='C0')
    ax[1].plot(np.ma.array(df.sa, mask=df['gradient_layer_final_mask']), df.p, color='C1')

    ax[0].legend()
    ax[0].invert_yaxis()

Let's try adding some random noise to the input data salinity

In [ ]:
arr_len = len(p_in)
rand_sample_a = (np.random.random(arr_len) - 0.5) * np.random.randint(2, size=arr_len) * np.random.randint(2, size=arr_len)  * np.random.randint(2, size=arr_len) 
sa_noise = sa_in + 0.03 * rand_sample_a
classify_test_data(p_in, ct_in, sa_noise)

Using default setttings, noisy salinity prevents the identification of salinity data. For this reason, we use the `temp_flag_only` kwarg to perform the preliminary classification with temperature only

In [ ]:
classify_test_data(p_in, ct_in, sa_noise, temp_flag_only=True)

As well as producing masks of final mixed layers and interfaces, the classifier returns the results of the classification steps and statistics on the mixed layers and interfaces

In [ ]:
df.head()

In [ ]:
mixes

In [ ]:
grads

# Classifier on Argo data

Here we use Argo data from the Meditteranean to test the classifier against a real world example.

In [ ]:
df_in = pd.read_csv('../data/vanderboog_argo_demo_data.csv')
df_out, mixes, grads = classify_staircase(df_in.pressure, df_in.conservative_temperature, df_in.absolute_salinity,
                                          temp_flag_only=True, show_steps=True)

Making two small changes to the kwargs, we obtain a better classification of this staircase 

In [ ]:
df_out, mixes, grads = classify_staircase(df_in.pressure, df_in.conservative_temperature, df_in.absolute_salinity,
                                          temp_flag_only=True, show_steps=True, layer_height_ratio=0.9, ml_grad=0.00048)

# Split into individual staircases

the function `identify_staircases_from_layers` takes the output from `classify_staircase` and groups the adjacent mixed layers and gradient layers together into individual staircases. The user can set the maximum distance between adjacent layers for them to cound in the same staricases. This can be useful with noisy data that may make small apparent breaks in staircases

In [ ]:
staircases_stats, staircases_full_dataframes = identify_staircases_from_layers(df_out, mixes, grads, show_plot=True)

Here's the summary stats from the largest staircase in this profile

In [ ]:
staircases_stats[3]

And here is the corresponding table of original data across this span of the profile

In [ ]:
staircases_full_dataframes[3]

### Help on the classifier

By calling `help()` on the classifier, we can view the docstring and see definitions of the variables that must be passed to it, and what the classifier kwargs are

In [ ]:
help(classify_staircase)